# Explicit-theta graph IRT: how A and S share item dependence

This notebook uses a VAE-based IRT model with an added item graph (called **VIF-model** here) as a **probe**, not as a proposed method.
The question is what goes wrong when the latent trait $\theta$ is defined explicitly, i.e. estimated for each person from all of their responses by an encoder, while a graph $S$ of direct item-to-item effects is estimated at the same time.

Two things can happen in such a model:

* **Self term**: an item's own response changes the encoder's $\theta$, which feeds back into that item's logit through $a_j$.
  Models that integrate $\theta$ out and work with conditional likelihoods, such as FLaG-IRT (Chen et al., 2018), have no such path.
* **Trade-off between $A$ and $S$**: the data determine only the total effect of item $k$ on item $j$. How that total is split between the $\theta$ path and the $S$ path
  is less constrained the denser the support of $S$.

Here VIF-model selects its graph by BIC along its own $\gamma$ path, and that solution is compared pair by pair with a fit given the true graph
(same data, same split, same starting point).

> Related experiments (including comparisons with FLaG-IRT on the same data-generating processes) have been run but are not public yet.

## Model

All responses are binary, $x_{ij}\in\{0,1\}$ (person $i$, item $j$), with a single factor ($K=1$).
$S$ is a symmetric item × item matrix with zero diagonal; $s_{jk}\neq 0$ means an edge (local dependence) between items $j$ and $k$.

| Model | Equation | Role |
|---|---|---|
| **VIF-model** | $\mathrm{logit}\,P(x_{ij}=1)=a_j\theta_i-d_j+\sum_{k\neq j}s_{jk}x_{ik}$ | probe model with explicit $\theta$ |
| **MIRT-only** (VAE-MIRT) | VIF-model with $S=0$ | graph-free reference · warm start |

* $\theta$ comes from an encoder (two-layer tanh MLP; amortized variational inference, Kingma & Welling, 2014; for VAE-based IRT see Curi et al., 2019; Wu et al., 2020; Urban & Bauer, 2021).
  The encoder takes the responses and observation indicators `[X, mask]` and outputs $q(\theta_i\mid x_i)=N(\mu_i,\sigma_i^2)$; the prior is $N(0,1)$.
* Objective $Q=(E_q[\mathrm{NLL}]+\mathrm{KL})/N+\gamma\lVert s\rVert_1$ (L1 penalty; Tibshirani, 1996). $A$, $d$ and the encoder are updated with Adam (Kingma & Ba, 2015);
  $s$ takes a step along the same gradient followed by soft-thresholding (a proximal step; Parikh & Boyd, 2014).

## What is measured

The change in item $j$'s logit when $x_k$ goes from 0 to 1 is the sum of two paths:

* **$\theta$ path** $P_{jk}$: $a_j$ × (average change in the encoder's posterior mean of $\theta$ when $x_k$ flips). The diagonal $P_{jj}$ is the self term.
* **$S$ path**: $s_{jk}$ (zero diagonal, so no self term).

**How to read the figures** (one panel per condition, one dot per item pair $(j, k)$)

* x-axis = change in the $\theta$-path effect (BIC-selected solution − true-graph fit), y-axis = change in the $S$-path effect.
* Dots on the line of slope −1: the total is unchanged and only its allocation moved.
* **Lower right** ($\theta$ path up, $S$ down): $A$ grows and $S$ turns negative by the same amount to cancel it; the self term grows as well.
* **Upper left** ($\theta$ path down, $S$ up): $S$ takes over dependence that $A$ carried.

## Data

24 items in testlets, $N=2000$ persons, one factor with evenly spread loadings. Three data-generating processes, each with a sparse and a dense true graph:

| Data | Generation | True edges (sparse / dense) |
|---|---|---|
| **VIF-type** | draw $\theta\sim N(0,1)$, then $x$ by Gibbs sampling from $\mathrm{logit}=\theta A^\top-d+XS$ (VIF-model is correctly specified) | 23 / 60 |
| **FLaG-type** | draw $(x, \theta)$ jointly from a FLaG-type joint distribution (Chen et al., 2018) (FLaG is correctly specified; the general factor is weaker) | 23 / 60 |
| **Bifactor** | testlet-specific nuisance factors (Gibbons & Hedeker, 1992; Bradlow, Wainer & Wang, 1999); both models are misspecified, loadings as in VIF-type | 24 / 60 same-testlet pairs |

One data set and one split per condition (a single seed), so the figures show direction, not size or reproducibility.
Deviations from the slope −1 line, or dots in an unexpected quadrant, are reported as they are.

## Settings

* Cell-level train 85 / test 15 split; all fits use the train cells only. Early stopping on convergence of the training objective (5 MC draws fixed at the start of the fit).
* $A$ starts at $|N(0,1)|$ (positive start, no constraint); graph fits start from the graph-free solution with $S=0$.
  The graph-free solution (MIRT-only) is the best of 5 seeds by training objective $Q$.
* $\gamma$ path and BIC selection follow Chen et al. (2018): an L1-regularized fit at each $\gamma$ on a grid → unpenalized refit for each distinct support (pairs with $|s_{jk}|>0.01$) →
  BIC $=2\,\mathrm{NLL}+(\text{number of parameters})\log N$ (Schwarz, 1978), where NLL is the marginal NLL with $\theta$ integrated out by quadrature.

## References

* Bradlow, E. T., Wainer, H., & Wang, X. (1999). A Bayesian random effects model for testlets. *Psychometrika*, 64(2). https://doi.org/10.1007/BF02294533
* Chen, Y., Li, X., Liu, J., & Ying, Z. (2018). Robust measurement via a fused latent and graphical item response theory model. *Psychometrika*, 83(3). https://doi.org/10.1007/s11336-018-9610-4
* Curi, M., Converse, G. A., Hajewski, J., & Oliveira, S. (2019). Interpretable variational autoencoders for cognitive models. *2019 International Joint Conference on Neural Networks (IJCNN)*. https://doi.org/10.1109/IJCNN.2019.8852333
* Gibbons, R. D., & Hedeker, D. R. (1992). Full-information item bi-factor analysis. *Psychometrika*, 57(3). https://doi.org/10.1007/BF02295430
* Kingma, D. P., & Ba, J. (2015). Adam: A method for stochastic optimization. *International Conference on Learning Representations (ICLR)*. https://arxiv.org/abs/1412.6980
* Kingma, D. P., & Welling, M. (2014). Auto-encoding variational Bayes. *International Conference on Learning Representations (ICLR)*. https://arxiv.org/abs/1312.6114
* Parikh, N., & Boyd, S. (2014). Proximal algorithms. *Foundations and Trends in Optimization*, 1(3). https://doi.org/10.1561/2400000003
* Schwarz, G. (1978). Estimating the dimension of a model. *The Annals of Statistics*, 6(2). https://doi.org/10.1214/aos/1176344136
* Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society: Series B*, 58(1). https://doi.org/10.1111/j.2517-6161.1996.tb02080.x
* Urban, C. J., & Bauer, D. J. (2021). A deep learning algorithm for high-dimensional exploratory item factor analysis. *Psychometrika*, 86(1). https://doi.org/10.1007/s11336-021-09748-3
* Wu, M., Davis, R. L., Domingue, B. W., Piech, C., & Goodman, N. (2020). Variational item response theory: Fast, accurate, and expressive. *Proceedings of the 13th International Conference on Educational Data Mining (EDM)*. https://arxiv.org/abs/2002.00276

## How to run

**Environment**: Python 3.10+, `torch`, `numpy`, `pandas`, `matplotlib`, `joblib` (CPU only).

```bash
pip install torch numpy pandas matplotlib joblib
```

**Data**: simulated inside the notebook; no files are needed.

**Runtime**: depends on the number of cores; the number of parallel workers can be set with the `IRT_WORKERS` environment variable.
We recommend first setting `SMOKE = True` to check that a small configuration runs end to end, then running the full experiment with `SMOKE = False`.

In [ ]:
SMOKE = False      # True runs a small configuration end to end (for checking the code)

DGP_N = 2000       # persons per data set
N_ITER = 3000      # max iterations per fit (early stopping usually kicks in before this)
if SMOKE:
    DGP_N, N_ITER = 800, 200

# **1. Libraries**

In [ ]:
import os

# Keep numpy / torch from spawning internal threads (parallelism comes only from parallel_map processes)
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_v, '1')

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

dtype = torch.float64
torch.set_default_dtype(dtype)
device = torch.device('cpu')          # matrices are small, so CPU is faster
torch.set_default_device(device)
torch.set_num_threads(1)


# Number of parallel processes (override with the IRT_WORKERS environment variable)
def _detect_workers():
    override = os.environ.get('IRT_WORKERS')
    if override:
        return max(1, int(override))
    try:
        n = len(os.sched_getaffinity(0))
    except AttributeError:
        n = os.cpu_count() or 4
    return max(1, min(n, 32))


SEARCH_WORKERS = _detect_workers()
print(f'SEARCH_WORKERS = {SEARCH_WORKERS}')

# **2. Data-generating processes**

In [ ]:
# VIF-type data: draw theta ~ N(0, I), then resample each item repeatedly from the conditional logit theta A' - d + X S (Gibbs sampling)
def gibbs_sample_X(theta, A, d, S_sym, n_burnin=60, seed=0):
    N, K = theta.shape
    J = A.shape[0]
    g = torch.Generator(device=device).manual_seed(int(seed))
    with torch.no_grad():
        base = theta @ A.T - d
        X = (torch.rand(N, J, generator=g) < 0.5).to(theta.dtype)
        for _ in range(n_burnin):
            for j in range(J):
                logit_j = base[:, j] + X @ S_sym[:, j]
                p = torch.sigmoid(logit_j)
                X[:, j] = (torch.rand(N, generator=g) < p).to(theta.dtype)
    return X


def _testlet_blocks(J, testlet_size):
    return [list(range(a, min(a + testlet_size, J))) for a in range(0, J, testlet_size)]


# True S with testlet structure. mode='bounded': degree <= max_degree within each testlet; 'dense': all pairs within each testlet
def make_S_testlet(J, testlet_size=4, mode='bounded', max_degree=2, weight=(0.4, 0.8), seed=0):
    rng = np.random.default_rng(seed)
    S = torch.zeros(J, J)
    for blk in _testlet_blocks(J, testlet_size):
        pairs = [(a, b) for ii, a in enumerate(blk) for b in blk[ii + 1:]]
        if mode == 'dense':
            chosen = pairs
        else:
            deg = {v: 0 for v in blk}
            rng.shuffle(pairs)
            chosen = []
            for a, b in pairs:
                if deg[a] < max_degree and deg[b] < max_degree:
                    chosen.append((a, b))
                    deg[a] += 1
                    deg[b] += 1
        for a, b in chosen:
            w = rng.uniform(*weight)
            S[a, b] = S[b, a] = w
    return S


# True A: simple structure; conc_sigma controls how concentrated loadings are within a column (column norm fixed at col_norm)
def make_A_concentrated(J, K, conc_sigma, col_norm=4.0, seed=0):
    rng = np.random.default_rng(seed)
    A = torch.zeros(J, K)
    for k in range(K):
        idx = [i for i in range(J) if i % K == k]
        mag = np.exp(rng.normal(0.0, conc_sigma, size=len(idx)))
        mag = mag / np.sqrt((mag ** 2).sum()) * col_norm
        A[idx, k] = torch.tensor(mag)
    return A


DGP_J, DGP_K = 24, 1
DGP_TESTLET = 6
A_CONC_SIGMA = 0.15          # evenly spread loadings
S_CONDITIONS = {'sparse': dict(mode='bounded', max_degree=2),
                'dense': dict(mode='dense')}
DGP_BURNIN = 20 if SMOKE else 60

_rng_d = np.random.default_rng(20260911)
d_true = torch.tensor(_rng_d.uniform(-1.0, 1.0, size=DGP_J))


def _data_seed(scond_key, rep, conditions=S_CONDITIONS):
    return 700_000 + 1000 * list(conditions).index(scond_key) + rep


def truth_vif(scond_key, seed=0):
    # col_norm 4 sqrt(2): per-item loading size equal to a two-factor design with 12 items and column norm 4 per factor
    A_t = make_A_concentrated(DGP_J, DGP_K, A_CONC_SIGMA, col_norm=4.0 * np.sqrt(2.0), seed=seed)
    S_t = make_S_testlet(DGP_J, testlet_size=DGP_TESTLET, weight=(0.4, 0.8),
                         seed=seed + 1, **S_CONDITIONS[scond_key])
    return A_t, S_t


def make_data_vif(scond_key, rep):
    seed = _data_seed(scond_key, rep)
    A_t, S_t = truth_vif(scond_key, seed=7)
    gen = torch.Generator(device=device).manual_seed(seed)
    theta_true = torch.randn(DGP_N, DGP_K, generator=gen)
    X = gibbs_sample_X(theta_true, A_t, d_true, S_t, n_burnin=DGP_BURNIN, seed=seed)
    mask_nan = torch.ones(DGP_N, DGP_J, dtype=torch.bool)
    splits = make_splits(mask_nan, gen=gen)
    return X, theta_true, A_t, S_t, splits

In [ ]:
# FLaG-type data: draw (x, theta) jointly from the FLaG-type joint distribution (inside Gibbs, theta | x ~ N(A'(x - 1/2), I)).
# Only the size of the true A and the burn-in differ from VIF-type data; the other constants (J, K, N, S conditions, d) are shared.
def gibbs_sample_flag(A, d, S_sym, N, n_burnin=200, seed=0):
    J, K = A.shape
    g = torch.Generator(device=device).manual_seed(int(seed))
    with torch.no_grad():
        X = (torch.rand(N, J, generator=g) < 0.5).to(A.dtype)
        for _ in range(n_burnin):
            theta = (X - 0.5) @ A + torch.randn(N, K, generator=g)
            base = theta @ A.T - d
            for j in range(J):
                p = torch.sigmoid(base[:, j] + (X - 0.5) @ S_sym[:, j])
                X[:, j] = (torch.rand(N, generator=g) < p).to(A.dtype)
        theta = (X - 0.5) @ A + torch.randn(N, K, generator=g)
    return X, theta


FLAG_BURNIN = 60 if SMOKE else 200
FLAG_COL_NORM = 1.6       # size of the true A (over all items). Larger values push FLaG-type responses to all-0 / all-1


def truth_flag(scond_key, seed=0):
    A_t = make_A_concentrated(DGP_J, DGP_K, A_CONC_SIGMA, col_norm=FLAG_COL_NORM, seed=seed)
    S_t = make_S_testlet(DGP_J, testlet_size=DGP_TESTLET, weight=(0.4, 0.8),
                         seed=seed + 1, **S_CONDITIONS[scond_key])
    return A_t, S_t


def make_data_flag(scond_key, rep):
    seed = _data_seed(scond_key, rep)
    A_t, S_t = truth_flag(scond_key, seed=7)
    gen = torch.Generator(device=device).manual_seed(seed)
    X, theta_true = gibbs_sample_flag(A_t, d_true, S_t, DGP_N, n_burnin=FLAG_BURNIN, seed=seed)
    mask_nan = torch.ones(DGP_N, DGP_J, dtype=torch.bool)
    splits = make_splits(mask_nan, gen=gen)
    return X, theta_true, A_t, S_t, splits

In [ ]:
# Bifactor data: logit_j = a_j theta_g + b_j theta_t(j) - d_j (testlet-specific nuisance factors).
# The true A and the other constants (J, K, N, d) are shared with VIF-type data. The S condition is the testlet size; the true graph is all same-testlet pairs.
def sample_bifactor_X(theta_g, theta_t, A, B, d, testlet_of, seed=0):
    g = torch.Generator(device=device).manual_seed(int(seed))
    with torch.no_grad():
        logits = theta_g @ A.T + theta_t[:, testlet_of] * B - d
        return (torch.rand(logits.shape, generator=g) < torch.sigmoid(logits)).to(A.dtype)


BIFACTOR_TESTLET = {'sparse': 3, 'dense': 6}      # testlet size: 8 testlets x 3 items (24 same-testlet pairs) / 4 x 6 (60 pairs)
BIFACTOR_NUIS = (np.sqrt(0.4), np.sqrt(0.8))      # range of nuisance loadings b_j: b_j b_k for same-testlet pairs lies in 0.4-0.8 (the edge-weight range)


def bifactor_params(scond_key, seed=0):
    A_t = make_A_concentrated(DGP_J, DGP_K, A_CONC_SIGMA, col_norm=4.0 * np.sqrt(2.0), seed=seed)
    testlet_of = torch.arange(DGP_J) // BIFACTOR_TESTLET[scond_key]
    B_t = torch.tensor(np.random.default_rng(seed + 1).uniform(*BIFACTOR_NUIS, size=DGP_J))
    return A_t, B_t, testlet_of


def truth_bifactor(scond_key, seed=0):
    A_t, B_t, testlet_of = bifactor_params(scond_key, seed=seed)
    same = (testlet_of[:, None] == testlet_of[None, :]) & ~torch.eye(DGP_J, dtype=torch.bool)
    return A_t, torch.outer(B_t, B_t) * same


def make_data_bifactor(scond_key, rep):
    seed = _data_seed(scond_key, rep, BIFACTOR_TESTLET)
    A_t, B_t, testlet_of = bifactor_params(scond_key, seed=7)
    S_t = truth_bifactor(scond_key, seed=7)[1]
    gen = torch.Generator(device=device).manual_seed(seed)
    theta_true = torch.randn(DGP_N, DGP_K, generator=gen)                        # general factor
    theta_t = torch.randn(DGP_N, int(testlet_of.max()) + 1, generator=gen)       # testlet nuisance factors
    X = sample_bifactor_X(theta_true, theta_t, A_t, B_t, d_true, testlet_of, seed=seed)
    mask_nan = torch.ones(DGP_N, DGP_J, dtype=torch.bool)
    splits = make_splits(mask_nan, gen=gen)
    return X, theta_true, A_t, S_t, splits

# **3. Function definitions**

Common utilities: parallel execution, cell-level splits, learning-rate schedule, handling S

In [ ]:
def _worker_init():
    import torch
    torch.set_default_dtype(torch.float64)
    torch.set_num_threads(1)


# Run several fits across processes. Each fit has its own seed, so results do not depend on parallelism.
_parallel_fail_shown = False   # print the fallback notice only once


def parallel_map(fn, items, max_workers=None):
    global _parallel_fail_shown
    items = list(items)
    if len(items) > 1:
        try:
            from joblib.externals.loky import get_reusable_executor
            pool = get_reusable_executor(max_workers=max_workers or SEARCH_WORKERS,
                                         initializer=_worker_init, timeout=600)
            return list(pool.map(fn, items))
        except (ImportError, RuntimeError, OSError, BlockingIOError) as e:
            if not _parallel_fail_shown:
                print(f'    ! parallel execution failed ({type(e).__name__}: {e}) -> falling back to sequential. (not shown again)')
                _parallel_fail_shown = True
    return [fn(x) for x in items]


# Randomly hold out a fraction frac of the observed cells (per cell, not per person)
def split_cells(mask, frac, gen=None):
    rand = torch.rand(mask.shape, dtype=torch.float32, device=mask.device, generator=gen)
    mask_held = mask & (rand < frac)
    mask_keep = mask & (rand >= frac)
    return mask_keep, mask_held


TEST_FRAC = 0.15          # train 85 / test 15


# Split observed cells into train / test (no validation set)
def make_splits(mask_nan, gen=None):
    mask_train, mask_test = split_cells(mask_nan, frac=TEST_FRAC, gen=gen)
    return dict(train_fit=mask_train, test_final=mask_test)


# Adam learning rate: linear warm-up for LR_WARMUP steps, halved after LR_PLATEAU_PATIENCE consecutive checks without improvement
LR_WARMUP, LR_PLATEAU_PATIENCE, LR_DECAY, LR_FLOOR_FRAC = 25, 4, 0.5, 0.05


class LRSchedule:
    def __init__(self, optimizer):
        self.scale, self.stall = 1.0, 0
        self._sched = torch.optim.lr_scheduler.LambdaLR(optimizer, self._factor)

    def _factor(self, t):
        return min((t + 1) / LR_WARMUP, 1.0) * self.scale

    def step(self):
        self._sched.step()

    def on_val(self, improved):
        if improved:
            self.stall = 0
            return
        self.stall += 1
        if self.stall % LR_PLATEAU_PATIENCE == 0:
            self.scale = max(self.scale * LR_DECAY, LR_FLOOR_FRAC)


S_INIT_SCALE = 0.01       # scale of the initial s_edge
A_INIT_SCALE = 1.0        # scale of the initial A


def init_s_edge(n, gen=None):
    return S_INIT_SCALE * torch.randn(n, generator=gen)


# S is learned only through the upper-triangular edge vector s_edge (zero diagonal, symmetric)
def build_symmetric_S(s_edge, J, iu, ju):
    S = torch.zeros(J, J, dtype=s_edge.dtype, device=s_edge.device)
    if s_edge.numel() > 0:
        S[iu, ju] = s_edge
        S[ju, iu] = s_edge
    return S


# Indices of candidate edges (i<j). If edge_mask is given, only allowed pairs
def edge_index(J, edge_mask=None):
    iu, ju = torch.triu_indices(J, J, offset=1)
    if edge_mask is not None:
        keep = edge_mask[iu, ju] > 0
        iu, ju = iu[keep], ju[keep]
    return iu, ju


# Resolve sign indeterminacy: flip A and theta together so that column means are positive
def orient_sign(A, Theta):
    signs = torch.sign(A.mean(dim=0))
    signs[signs == 0] = 1.0
    D = torch.diag(signs)
    return A @ D, Theta @ D

VIF-model (theta estimated by an encoder)

In [ ]:
N_MC_SAMPLES = 5          # theta samples drawn per iteration


# encoder: responses -> posterior mean and log-variance of theta.
# Input is [X, mask] (2J dims): responses (0 if unobserved) and observation indicators. Item parameters A, d, S are not inputs.
class ThetaEncoder(nn.Module):
    def __init__(self, J, K, hidden=32):
        super().__init__()
        self.K = K
        self.net = nn.Sequential(
            nn.Linear(2 * J, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2 * K),
        )

    def build_input(self, X_filled, observed_mask):
        return torch.cat([X_filled, observed_mask.to(X_filled.dtype)], dim=1)

    def forward(self, enc_input):
        out = self.net(enc_input)
        mu, log_var = out[:, :self.K], out[:, self.K:]
        log_var = torch.clamp(log_var, min=-8.0, max=8.0)
        return mu, log_var


# Redraw encoder weights from this fit's own RNG (same distribution as the default nn.Linear init)
def reseed_encoder_(encoder, gen):
    with torch.no_grad():
        for m in encoder.modules():
            if isinstance(m, nn.Linear):
                bound = float(m.weight.shape[1]) ** -0.5
                m.weight.uniform_(-bound, bound, generator=gen)
                if m.bias is not None:
                    m.bias.uniform_(-bound, bound, generator=gen)
    return encoder


def score_theta_vif(X, observed_mask, A, encoder):
    with torch.no_grad():
        X_filled = torch.where(observed_mask, X, torch.zeros_like(X))
        mu, _ = encoder(encoder.build_input(X_filled, observed_mask))
    return mu


# Integrals over theta are computed by quadrature (K=1 only).
#   GH_Z, GH_W       : Gauss-Hermite nodes for expectations under q = N(mu, sigma^2) (theta = mu + sigma * z)
#   TH_GRID, TH_LOGW : equally spaced grid (step 0.05) and log weights for integrating against the prior N(0, 1)
_gh_z, _gh_w = np.polynomial.hermite_e.hermegauss(21)
GH_Z, GH_W = torch.tensor(_gh_z), torch.tensor(_gh_w / _gh_w.sum())
TH_GRID = torch.linspace(-8.0, 8.0, 321)
TH_LOGW = -0.5 * TH_GRID ** 2 - 0.5 * np.log(2 * np.pi) + np.log(float(TH_GRID[1] - TH_GRID[0]))


# Sum of NLL over observed cells for each person x theta node.  theta: (n, Q),  offset = X S - d: (n, J)  ->  (n, Q)
def nll_on_theta_points(theta, A, offset, X_zero, obs_f):
    assert A.shape[1] == 1, 'theta quadrature supports K=1 only'
    logits = theta.unsqueeze(-1) * A[:, 0] + offset.unsqueeze(1)
    nll = F.binary_cross_entropy_with_logits(logits, X_zero.unsqueeze(1).expand_as(logits), reduction='none')
    return (nll * obs_f.unsqueeze(1)).sum(dim=-1)


# E_q[NLL] summed over persons (q = N(mu, std^2))
def expected_nll(mu, std, A, offset, X_zero, obs_f):
    return float((nll_on_theta_points(mu + std * GH_Z, A, offset, X_zero, obs_f) * GH_W).sum())


# Marginal NLL = - sum_i log ∫ exp(-NLL_i(theta)) N(theta; 0, 1) dtheta.  Used as the fit term of BIC.
def marginal_nll(A, offset, X_zero, obs_f, chunk=500):
    total = 0.0
    for a in range(0, X_zero.shape[0], chunk):
        sl = slice(a, a + chunk)
        nll = nll_on_theta_points(TH_GRID.expand(X_zero[sl].shape[0], -1), A, offset[sl], X_zero[sl], obs_f[sl])
        total -= float(torch.logsumexp(TH_LOGW - nll, dim=1).sum())
    return total


# VIF-model:  logits = theta A' - d + X S,  theta ~ q(theta | x) = N(mu, sigma^2) (encoder)
#   objective Q = (E_q[NLL] + KL) / N + gamma * |s_edge|_1
#   A, d, encoder use Adam; s_edge takes an lr_S step along the gradient from the same backward pass, then soft-threshold (threshold lr_S * gamma)
#   early stopping: every check_every steps, evaluate Q on the train cells (E_q from fixed MC draws); after patience checks without improvement, restore the best state
#   if init is given, start from that solution (warm start). If init has no matching s_edge, S starts at 0.
#   fit_graph=False gives S = 0 (graph-free MIRT)
def fit_vif(X, observed_mask, K, gamma=0.0, fit_graph=True, edge_mask=None,
            n_iter=1500, lr=0.05, lr_S=0.05, hidden=16, check_every=25, patience=16,
            init=None, gen=None):
    N, J = X.shape
    iu, ju = (edge_index(J, edge_mask) if fit_graph
              else (torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)))

    if init is None:
        # A starts positive. Starting with mixed signs can trap the fit in a local solution with split loading signs (not a constraint).
        A = (A_INIT_SCALE * torch.randn(J, K, generator=gen)).abs().requires_grad_(True)
        d = torch.zeros(J, requires_grad=True)
        encoder = reseed_encoder_(ThetaEncoder(J, K, hidden=hidden).to(dtype=X.dtype), gen)
        s_edge = init_s_edge(iu.numel(), gen).requires_grad_(True)
    else:
        A = init['A'].clone().requires_grad_(True)
        d = init['d'].clone().requires_grad_(True)
        encoder = ThetaEncoder(J, K, hidden=hidden).to(dtype=X.dtype)
        encoder.load_state_dict(init['encoder'])
        s_prev = init.get('s_edge')
        s_edge = ((s_prev.clone() if (s_prev is not None and s_prev.numel() == iu.numel())
                   else torch.zeros(iu.numel())).requires_grad_(True))

    X_zero = torch.where(observed_mask, X, torch.zeros_like(X))
    obs_f = observed_mask.to(X.dtype)

    # 0/1 mask of which loadings are estimated. All loadings are free for now (change this for a confirmatory structure).
    Lambda_mask = torch.ones(J, K)

    def _A_eff(A_):
        return A_ * Lambda_mask

    optimizer = torch.optim.Adam([A, d] + list(encoder.parameters()), lr=lr)
    scheduler = LRSchedule(optimizer)
    best_loss, best_state, stall = float('inf'), None, 0
    # Noise for the early-stopping theta draws. Drawn once and reused at every check (so comparisons between checks are free of MC noise)
    check_eps = torch.randn(N_MC_SAMPLES, N, K, dtype=X.dtype, generator=gen)

    for t in range(n_iter):
        optimizer.zero_grad(set_to_none=True)
        S_sym = build_symmetric_S(s_edge, J, iu, ju)

        mu, log_var = encoder(encoder.build_input(X_zero, observed_mask))
        std = (0.5 * log_var).exp()
        mc_eps = torch.randn(N_MC_SAMPLES, *mu.shape, dtype=mu.dtype, generator=gen)
        theta_samples = mu.unsqueeze(0) + std.unsqueeze(0) * mc_eps

        logits_mc = theta_samples @ _A_eff(A).T - d + (X_zero @ S_sym).unsqueeze(0)
        nll_elem = F.binary_cross_entropy_with_logits(
            logits_mc, X_zero.unsqueeze(0).expand_as(logits_mc), reduction='none')
        nll_sum = (nll_elem * obs_f.unsqueeze(0)).sum() / N_MC_SAMPLES
        kl = 0.5 * (std ** 2 + mu ** 2 - 1 - log_var).sum()

        loss = (nll_sum + kl) / N
        loss.backward()
        optimizer.step()
        scheduler.step()

        # s_edge: plain gradient step + soft-threshold (the L1 penalty is not differentiated; it is applied only here)
        if fit_graph and s_edge.numel() > 0:
            with torch.no_grad():
                gs = s_edge.grad
                if gs is not None:
                    z = s_edge - lr_S * gs
                    s_edge.copy_(torch.sign(z) * torch.clamp(z.abs() - lr_S * gamma, min=0.0))
                    s_edge.grad = None

        # Early stopping: stop when Q = E_q[NLL] + KL (+ penalty) on the train cells stops improving. E_q is the average over draws built from check_eps
        if t % check_every == 0:
            with torch.no_grad():
                S_cur = build_symmetric_S(s_edge, J, iu, ju)
                mu_v, lv_v = encoder(encoder.build_input(X_zero, observed_mask))
                theta_v = mu_v.unsqueeze(0) + (0.5 * lv_v).exp().unsqueeze(0) * check_eps
                logits_v = theta_v @ _A_eff(A).T - d + (X_zero @ S_cur).unsqueeze(0)
                cur_loss = (F.binary_cross_entropy_with_logits(
                    logits_v, X_zero.unsqueeze(0).expand_as(logits_v), reduction='none')
                    * obs_f.unsqueeze(0)).sum().item() / N_MC_SAMPLES
                cur_loss += (0.5 * (lv_v.exp() + mu_v ** 2 - 1 - lv_v).sum().item()
                             + N * gamma * s_edge.abs().sum().item())
            scheduler.on_val(cur_loss < best_loss)
            if cur_loss < best_loss:
                best_loss = cur_loss
                best_state = (A.detach().clone(), s_edge.detach().clone(), d.detach().clone(),
                              {k: v.detach().clone() for k, v in encoder.state_dict().items()})
                stall = 0
            else:
                stall += 1
                if stall >= patience:
                    break

    A, s_edge, d, enc_state = best_state
    encoder.load_state_dict(enc_state)

    with torch.no_grad():
        A = _A_eff(A)
        S_final = build_symmetric_S(s_edge, J, iu, ju)
        mu_final, log_var_final = encoder(encoder.build_input(X_zero, observed_mask))
        std_final = (0.5 * log_var_final).exp()
        kl_final = 0.5 * (std_final ** 2 + mu_final ** 2 - 1 - log_var_final).sum()
        offset_final = X_zero @ S_final - d
        nll_final = expected_nll(mu_final, std_final, A, offset_final, X_zero, obs_f)
        Q_final = ((nll_final + kl_final) / N + gamma * s_edge.abs().sum()).item()
        marg_nll = marginal_nll(A, offset_final, X_zero, obs_f)

    diag = dict(n_edges=int((s_edge.abs() > 1e-3).sum().item()),
                n_iter_run=t + 1,
                marg_nll=marg_nll,      # marginal NLL with theta integrated out (sum over persons). Used by BIC
                # solution that can be passed as init to another fit (warm start)
                state=dict(A=A.detach().clone(), d=d.detach().clone(),
                           s_edge=s_edge.detach().clone(),
                           encoder={k: v.detach().clone() for k, v in encoder.state_dict().items()}))

    return A.detach(), S_final.detach(), mu_final.detach(), d.detach(), Q_final, encoder, diag

Model fitting: every fit returns results in the same format

In [ ]:
LR = 0.05                 # Adam learning rate for all fits
# Graph model and its "graph-free" counterpart. The counterpart's solution is the warm start and the reference for the |A| ratio.
REF_OF = {'VIF-model': 'MIRT-only'}


# Fit one model and return it in a common format: A, S (symmetric, zero diagonal), d, theta_train, diag, state
#   name : 'VIF-model' / 'MIRT-only' (VIF without S)
#   Every fit trains on train_fit cells only and stops on convergence of the training objective. theta is estimated with train_fit cells as context.
def run_model(name, X, splits, K, edge_mask=None, gamma=0.0, n_iter=800, lr=LR, hidden=16,
              init=None, fit_seed=None):
    gen = torch.Generator(device=device).manual_seed(int(fit_seed))
    mtf = splits['train_fit']
    A_raw, S, mu, d, Q_final, enc, diag = fit_vif(
        X, mtf, K, gamma=gamma, fit_graph=(name != 'MIRT-only'), edge_mask=edge_mask,
        n_iter=n_iter, lr=lr, hidden=hidden, init=init, gen=gen)
    S_sym = 0.5 * (S + S.T)
    A_or, _ = orient_sign(A_raw, mu)
    _, theta_train = orient_sign(A_raw, score_theta_vif(X, mtf, A_raw, enc))
    diag['Q_final'] = float(Q_final)
    return dict(name=name, A=A_or, S=S_sym, d=d, theta_train=theta_train, diag=diag,
                state=diag.get('state'))

VIF-model's own gamma path and BIC selection

In [ ]:
GAMMA_FRAC = (0.10, 1.05)   # gamma grid range (relative to gamma_max)
N_GAMMA = 20                # number of grid points
N_REF_STARTS = 5            # number of graph-free reference fits with different seeds (the one with the lowest training Q is used)
SUPPORT_TOL = 0.01          # in regularized fits, only pairs with |s| above this count as support (excludes values flickering near the threshold)
if SMOKE:
    N_GAMMA = 14
    N_REF_STARTS = 2


def edge_support(S_sym, tol=1e-3):
    J = S_sym.shape[0]
    iu, ju = torch.triu_indices(J, J, offset=1)
    return (S_sym[iu, ju].abs() > tol).cpu().numpy()


# Smallest gamma at which the L1 term turns the graph off entirely at S = 0: |d(NLL/N)/dS|_inf. Used to scale the gamma grid.
def gamma_max_of(X, splits, ref):
    N, J = X.shape
    mtf = splits['train_fit']
    X_zero = torch.where(mtf, X, torch.zeros_like(X))
    iu, ju = torch.triu_indices(J, J, offset=1)
    s0 = torch.zeros(iu.numel(), requires_grad=True)
    S_off = build_symmetric_S(s0, J, iu, ju)
    base = ref['theta_train'] @ ref['A'].T - ref['d']
    logits = base + X_zero @ S_off
    nll = (F.binary_cross_entropy_with_logits(logits, X_zero, reduction='none')
           * mtf.to(X.dtype)).sum() / N
    return float(torch.autograd.grad(nll, s0)[0].abs().max().item())


# Graph-free solution: warm start for every candidate, scale of the gamma grid, reference for the |A| ratio
#   Depending on initialization, the graph-free fit (MIRT-only) can end in a solution where theta is switched off (A ~ 0). So we fit
#   N_REF_STARTS seeds and keep the lowest training Q (the first start uses fit_seed as is).
def graph_free_reference(name, X, splits, K, n_iter, fit_seed=0, hidden=16):
    seeds = [fit_seed + 7919 * k for k in range(N_REF_STARTS)]

    def _start(seed):
        return run_model(REF_OF[name], X, splits, K, gamma=0.0, n_iter=n_iter, lr=LR, hidden=hidden,
                         fit_seed=seed)

    fits = parallel_map(_start, seeds, max_workers=SEARCH_WORKERS)
    res = min(fits, key=lambda r: r['diag']['Q_final'])
    res['diag']['ref_starts_Q'] = [r['diag']['Q_final'] for r in fits]      # training Q per start (for inspection)
    ref = dict(res=res, state=res['state'], Q=res['diag']['Q_final'],
               A_norm=float(torch.linalg.norm(res['A'])),
               A=res['A'], d=res['d'], theta_train=res['theta_train'])
    ref['gamma_max'] = gamma_max_of(X, splits, ref)
    return ref


# Dedicated seed per candidate
def _fit_seed_for(gamma, tag=0):
    return (int(round(float(gamma) * 1e7)) * 131 + int(tag) * 7919) % (2 ** 31)


def gamma_grid_of(ref):
    return np.geomspace(GAMMA_FRAC[0] * ref['gamma_max'],
                        GAMMA_FRAC[1] * ref['gamma_max'], N_GAMMA)


# Number of parameters: loadings J*K + intercepts J + edges
def n_params_of(name, J, K, n_edges):
    return J * K + J + int(n_edges)


def support_to_mask(support, J):
    iu, ju = torch.triu_indices(J, J, offset=1)
    v = torch.as_tensor(support, dtype=dtype)
    m = torch.zeros(J, J, dtype=dtype)
    m[iu, ju] = v
    m[ju, iu] = v
    return m


# Gamma path for one model (procedure of Chen et al., 2018). Returns (path, sel)
#   1) L1-regularized fit at each gamma on the grid (warm start from the graph-free solution ref) -> support {|s_ij| > SUPPORT_TOL}
#   2) for each distinct support, an unpenalized refit allowing only that support
#   3) for each candidate: BIC = 2 NLL_refit + (number of parameters) log N, with NLL_refit the marginal NLL (theta integrated by quadrature)
#   4) sel = minimum BIC over all candidates (ties go to the sparser one)
def gamma_path(name, X, splits, K, n_iter, ref, fit_seed=0, hidden=16):
    N, J = X.shape
    n_persons = int(splits['train_fit'].any(dim=1).sum().item())
    grid = gamma_grid_of(ref)
    kw = dict(n_iter=n_iter, lr=LR, hidden=hidden)

    def _reg(g):
        r = run_model(name, X, splits, K, edge_mask=None, gamma=float(g),
                      init=ref['state'], fit_seed=_fit_seed_for(g, fit_seed), **kw)
        return dict(gamma=float(g), reg=r, support=edge_support(r['S'], tol=SUPPORT_TOL))

    path = sorted(parallel_map(_reg, grid, max_workers=SEARCH_WORKERS), key=lambda p: p['gamma'])

    # Refit each distinct support only once
    uniq = {}
    for p in path:
        uniq.setdefault(p['support'].tobytes(), p)

    def _refit(p):
        mask = support_to_mask(p['support'], J)
        iu, ju = edge_index(J, mask)
        init = dict(p['reg']['state'])
        init['s_edge'] = p['reg']['S'][iu, ju].clone()
        r = run_model(name, X, splits, K, edge_mask=mask, gamma=0.0, init=init,
                      fit_seed=_fit_seed_for(p['gamma'], fit_seed + 1), **kw)
        return p['support'].tobytes(), r

    refits = dict(parallel_map(_refit, list(uniq.values()), max_workers=SEARCH_WORKERS))

    for p in path:
        r = refits[p['support'].tobytes()]
        p['n_edges'] = int(p['support'].sum())
        p['Q_refit'] = r['diag']['Q_final']
        p['nll_refit'] = r['diag']['marg_nll']
        p['bic'] = 2.0 * p['nll_refit'] + n_params_of(name, J, K, p['n_edges']) * np.log(n_persons)
        del p['reg']

    sel = min(path, key=lambda p: (p['bic'], -p['gamma']))      # ties go to the sparser one
    for p in path:
        p['selected'] = p is sel
    return path, sel

Path decomposition: split each logit change into the theta path and the S path, and compare the BIC-selected solution with the true-graph fit

In [ ]:
# Split "change in item j's logit when x_k goes 0 -> 1" into the theta path P[j, k] and the S path S[j, k] (row j, column k)
#   P[j, k] = a_j x (change in the encoder's posterior mean of theta, averaged over persons who observed item k).  The diagonal P[j, j] is the self term
def path_effects(res, X, mask, hidden=16):
    J = X.shape[1]
    st = res['state']
    assert st['A'].shape[1] == 1, 'K=1 only'
    enc = ThetaEncoder(J, 1, hidden=hidden).to(dtype=X.dtype)
    enc.load_state_dict(st['encoder'])
    X_zero = torch.where(mask, X, torch.zeros_like(X))
    delta = torch.zeros(J)
    with torch.no_grad():
        for k in range(J):
            rows = mask[:, k]
            x1, x0 = X_zero[rows].clone(), X_zero[rows].clone()
            x1[:, k], x0[:, k] = 1.0, 0.0
            mu1, _ = enc(enc.build_input(x1, mask[rows]))
            mu0, _ = enc(enc.build_input(x0, mask[rows]))
            delta[k] = (mu1 - mu0).mean()
    return st['A'] @ delta.reshape(1, J), res['S']


# Compare the two paths of the BIC-selected solution (res) and the true-graph fit (res_known) pair by pair. "non-edge" = pairs where the true S is 0.
#   self_term*                              : self term averaged over items
#   theta_cross* / S_cross* / total_cross*  : mean theta path / S path / their sum over non-edge pairs
#   exchange_slope / exchange_corr          : slope and correlation from regressing the change in the S path on the change in the theta path over all pairs (-1 if the total is unchanged)
def exchange_metrics(res, res_known, X, mask, S_true):
    P, S = path_effects(res, X, mask)
    Pk, Sk = path_effects(res_known, X, mask)
    J = S.shape[0]
    off = ~torch.eye(J, dtype=torch.bool)
    edge = off & (S_true.abs() > 0)
    non = off & ~edge
    dP, dS = (P - Pk)[off], (S - Sk)[off]
    cP, cS = dP - dP.mean(), dS - dS.mean()
    rec = dict(self_term=torch.diagonal(P).mean().item(), self_term_known=torch.diagonal(Pk).mean().item(),
               theta_cross=P[non].mean().item(), theta_cross_known=Pk[non].mean().item(),
               S_cross=S[non].mean().item(), S_cross_known=Sk[non].mean().item(),
               total_cross=(P + S)[non].mean().item(), total_cross_known=(Pk + Sk)[non].mean().item(),
               exchange_slope=((cP * cS).sum() / (cP ** 2).sum()).item(),
               exchange_corr=((cP * cS).sum() / torch.sqrt((cP ** 2).sum() * (cS ** 2).sum())).item())
    return rec, dict(dP=P - Pk, dS=S - Sk, edge=edge, non=non)

# **4. Experiment: for each condition, VIF-model's gamma path, the BIC-selected solution and the true-graph fit**

In [ ]:
# (data name, S condition, generator). Loadings are evenly spread in every case.
CASES = [('VIF-type data', 'sparse', make_data_vif), ('VIF-type data', 'dense', make_data_vif),
         ('FLaG-type data', 'sparse', make_data_flag), ('FLaG-type data', 'dense', make_data_flag),
         ('Bifactor data', 'sparse', make_data_bifactor), ('Bifactor data', 'dense', make_data_bifactor)]
REP, FIT_SEED = 0, 9000          # data replicate and fitting seed


# Rebuild the refit of the candidate selected on the gamma path (gamma_path does not return refits).
# It uses the same starting point and seed as on the path, so it is the solution that produced that BIC (checked below).
def selected_refit(X, splits, K, n_iter, ref, path, sel, fit_seed=0, hidden=16):
    J = X.shape[1]
    g = min(p['gamma'] for p in path if p['support'].tobytes() == sel['support'].tobytes())
    kw = dict(n_iter=n_iter, lr=LR, hidden=hidden)
    reg = run_model('VIF-model', X, splits, K, edge_mask=None, gamma=float(g),
                    init=ref['state'], fit_seed=_fit_seed_for(g, fit_seed), **kw)
    mask = support_to_mask(sel['support'], J)
    iu, ju = edge_index(J, mask)
    init = dict(reg['state'])
    init['s_edge'] = reg['S'][iu, ju].clone()
    return run_model('VIF-model', X, splits, K, edge_mask=mask, gamma=0.0, init=init,
                     fit_seed=_fit_seed_for(g, fit_seed + 1), **kw)


t0 = time.time()
rows, keep_all = [], {}
for data_name, sk, make in CASES:
    X, theta_true, A_t, S_t, splits = make(sk, REP)
    mtf = splits['train_fit']
    n_persons = int(mtf.any(dim=1).sum().item())
    ref = graph_free_reference('VIF-model', X, splits, DGP_K, N_ITER, fit_seed=FIT_SEED)
    path, sel = gamma_path('VIF-model', X, splits, DGP_K, N_ITER, ref, fit_seed=FIT_SEED)
    res_bic = selected_refit(X, splits, DGP_K, N_ITER, ref, path, sel, fit_seed=FIT_SEED)
    bic_again = (2.0 * res_bic['diag']['marg_nll']
                 + n_params_of('VIF-model', DGP_J, DGP_K, sel['n_edges']) * np.log(n_persons))
    true_mask = (S_t.abs() > 1e-12).to(dtype)
    res_known = run_model('VIF-model', X, splits, DGP_K, edge_mask=true_mask, gamma=0.0, n_iter=N_ITER,
                          lr=0.05, hidden=16, init=ref['state'], fit_seed=FIT_SEED)

    rec, keep = exchange_metrics(res_bic, res_known, X, mtf, S_t)
    sel_mask = support_to_mask(sel['support'], DGP_J) > 0
    n_true = int((S_t.abs() > 0).sum().item() // 2)
    rec.update({'data': data_name, 'S_condition': sk, 'n_true_edges': n_true, 'n_sel_edges': sel['n_edges'],
                'n_sel_true': int((sel_mask & (S_t.abs() > 0)).sum().item() // 2),
                'bic_sel': sel['bic'], 'bic_refit_again': bic_again,
                'A_ratio_bic': float(torch.linalg.norm(res_bic['A'])) / ref['A_norm'],
                'A_ratio_known': float(torch.linalg.norm(res_known['A'])) / ref['A_norm']})
    rows.append(rec)
    keep_all[(data_name, sk)] = keep
    print(f"  [{data_name} / {sk}] selected {sel['n_edges']} edges (true {n_true}), BIC {sel['bic']:.1f} "
          f"(rebuilt refit {bic_again:.1f})  edges along the path {[p['n_edges'] for p in path]}  ({time.time() - t0:.0f}s)")

df_res = pd.DataFrame(rows).set_index(['data', 'S_condition'])

# **5. Results**

Summary table

In [ ]:
print('=== BIC-selected solution (bic) vs true-graph fit (known) ===')
print(df_res[['n_true_edges', 'n_sel_edges', 'n_sel_true', 'A_ratio_known', 'A_ratio_bic']].round(3).to_string())
print()
print(df_res[['self_term_known', 'self_term', 'theta_cross_known', 'theta_cross', 'S_cross_known', 'S_cross',
              'total_cross_known', 'total_cross', 'exchange_slope', 'exchange_corr']].round(3).to_string())
print()
print('A_ratio_* = loading size relative to the graph-free solution (|A| / |A_ref|). *_cross = mean over pairs that are not true edges.')
print('exchange_slope = slope from regressing the change in the S path on the change in the theta path (-1 if the total is unchanged).')
print('Difference between bic_sel and bic_refit_again (should be 0 so that the plotted solution is the one that produced the BIC):',
      (df_res['bic_refit_again'] - df_res['bic_sel']).abs().round(3).tolist())

Figure per condition: VIF-type and FLaG-type data (rows = data-generating process, columns = density of the true S)

In [ ]:
# One panel per condition: for each pair, the change in the theta path (x) and in the S path (y). Change = BIC-selected solution - true-graph fit.
# The box shows true-graph fit -> BIC-selected solution. The other numbers (selected edges, slope, ...) are in the table above.
#   datas: data names to place as rows (columns are S conditions)
def condition_figure(datas):
    sks = list(dict.fromkeys(s for _, s, _ in CASES))
    fig, axes = plt.subplots(len(datas), len(sks), figsize=(4.6 * len(sks), 4.6 * len(datas)), squeeze=False)
    for i, data_name in enumerate(datas):
        for j, sk in enumerate(sks):
            ax, k, r = axes[i, j], keep_all[(data_name, sk)], df_res.loc[(data_name, sk)]
            lim = 1.08 * max(float(k['dP'][k['edge'] | k['non']].abs().max()),
                             float(k['dS'][k['edge'] | k['non']].abs().max()), 1e-6)
            ax.plot([-lim, lim], [lim, -lim], 'k--', lw=0.8, label='slope -1')
            ax.axhline(0.0, color='0.6', lw=0.6)
            ax.axvline(0.0, color='0.6', lw=0.6)
            ax.plot(k['dP'][k['non']].numpy(), k['dS'][k['non']].numpy(), '.', color='0.45', ms=3, label='non-edge pairs')
            ax.plot(k['dP'][k['edge']].numpy(), k['dS'][k['edge']].numpy(), '.', color='C3', ms=6, label='true edges')
            ax.text(0.97, 0.97,
                    f"|A| / |A_ref|: {r['A_ratio_known']:.2f} -> {r['A_ratio_bic']:.2f}\n"
                    f"self term: {r['self_term_known']:.2f} -> {r['self_term']:.2f}",
                    transform=ax.transAxes, ha='right', va='top', fontsize=10,
                    bbox=dict(boxstyle='round', facecolor='white', edgecolor='0.7', alpha=0.9))
            ax.set_xlim(-lim, lim)
            ax.set_ylim(-lim, lim)
            ax.set_aspect('equal')
            ax.set_title(f'{data_name}, {sk}', fontsize=11)
            ax.tick_params(labelsize=10)
            ax.grid(alpha=0.3)
            if i == len(datas) - 1:
                ax.set_xlabel('change in theta-path effect', fontsize=11)
            if j == 0:
                ax.set_ylabel('change in S-path effect', fontsize=11)
    axes[0][0].legend(fontsize=9, loc='lower left')
    fig.tight_layout()
    plt.show()


condition_figure(['VIF-type data', 'FLaG-type data'])

Figure per condition: VIF-type and bifactor data

In [ ]:
condition_figure(['VIF-type data', 'Bifactor data'])

**How to read**

* The `known -> BIC-selected` values in each box give the direction. If `|A| / |A_ref|` and the self term both grow and the dots sit in the lower right,
  $S$ cancels $A$'s cross terms with negative values and lets $A$ (and the self term) grow. If `|A| / |A_ref|` shrinks and the dots sit in the upper left, $S$ has taken over dependence that $A$ explained.
* How closely the dots follow the slope −1 line (`exchange_slope`, `exchange_corr` in the table) is the evidence for "the total is unchanged, only the allocation moved". Departures from the line mean the total dependence itself changed.
* The reference (true-graph fit) is itself an estimate. If `A_ratio_known` is far from 1, the reference itself has moved, so read that panel with care.
* This is a single seed: it can show direction, but not size or reproducibility.